In [ ]:
# Fig 1 Performance of single-start and multi-start coarse-to-fine ICP alignment

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from config import config

rmsd_standard = np.load(config.directory.analysis / "standard_alignment_0_4.npy").flatten()
rmsd_heuristic = np.load(config.directory.analysis / "heuristic_alignment_0_4.npy").flatten()
df = pd.read_csv(config.directory.analysis / "alignment_performance.csv")

bin_width = 0.1
bins_standard = np.arange(0, max(rmsd_standard), bin_width)
bins_heuristic = np.arange(0, max(rmsd_heuristic), bin_width)

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(7.5, 6))

# Subfigure A: RMSD distribution of standard alignment
sns.histplot(ax=axes[0, 0], data=rmsd_standard, bins=bins_standard, color="0.6", stat="probability")
axes[0, 0].set_xlabel("RMSD", fontsize=10)
axes[0, 0].set_ylabel("Relative frequency", fontsize=10)
axes[0, 0].set_title("Single ICP alignment", fontsize=10, fontweight="bold")
axes[0, 0].set_xlim(0, 2.5)
for patch in axes[0, 0].patches:
    patch.set_edgecolor("black")
    patch.set_linewidth(0.5)

# Subfigure B: RMSD distribution of heuristic alignment
sns.histplot(ax=axes[0, 1], data=rmsd_heuristic, bins=bins_heuristic, color="0.6", stat="probability")
axes[0, 1].set_xlabel("RMSD", fontsize=10)
axes[0, 1].set_ylabel("Relative frequency", fontsize=10)
axes[0, 1].set_title("Two-stage multi-start heuristic", fontsize=10, fontweight="bold")
axes[0, 1].set_xlim(0, 2.5)
for patch in axes[0, 1].patches:
    patch.set_edgecolor("black")
    patch.set_linewidth(0.5)

# Subfigure C: Success rate vs. N
sns.lineplot(ax=axes[1, 0], x="N", y="Successful", data=df, color="black")
axes[1, 0].set_xlabel("$T$", fontsize=12)
axes[1, 0].set_ylabel("Success rate", fontsize=10)
axes[1, 0].set_ylim(0.3, 1.05)
axes[1, 0].axvline(x=200, color="black", linestyle="--", linewidth=1)

# Subfigure D: Runtime vs. N
sns.lineplot(ax=axes[1, 1], x="N", y="Runtime", data=df, color="black")
axes[1, 1].set_xlabel("$T$", fontsize=12)
axes[1, 1].set_ylabel("Time [s/alignment]", fontsize=10)
axes[1, 1].axvline(x=200, color="black", linestyle="--", linewidth=1)

subplot_labels = ["A", "B", "C", "D"]
for ax, label in zip(axes.flat, subplot_labels):
    ax.text(-0.1, 1.1, label, transform=ax.transAxes, fontsize=16, fontweight="bold", va="top", ha="right")

plt.tight_layout()
plt.savefig(config.directory.figures / "manuscript/Fig1.png", dpi=600)
plt.savefig(config.directory.figures / "manuscript/Fig1.tif", dpi=600)
plt.show()

In [ ]:
# Fig 2 Distribution of metals and enzyme classes before and after redundancy filtering

import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
import pandas as pd
from config import config

# --- EC distributions ---
path_ec_filtered = config.directory.analysis / "ec_distribution_filtered.npz"
path_ec_full = config.directory.analysis / "ec_distribution_full.npz"

data_ec_filtered = np.load(path_ec_filtered)
data_ec_full = np.load(path_ec_full)

ec_classes = data_ec_filtered["classes"]
ec_distribution = data_ec_filtered["dataset"]
pdb_distribution = data_ec_full["dataset"]

df_dataset = pd.DataFrame({
    "EC": [f"EC {int(i)}" for i in ec_classes],
    "Frequency": ec_distribution,
    "Source": "Filtered"
})
df_pdb = pd.DataFrame({
    "EC": [f"EC {int(i)}" for i in ec_classes],
    "Frequency": pdb_distribution,
    "Source": "Retrieved"
})
df_combined = pd.concat([df_dataset, df_pdb])

# --- Ligand distributions ---
path_filtered = config.directory.analysis / "ligand_distribution_filtered.npz"
path_full = config.directory.analysis / "ligand_distribution_full.npz"

data_filtered = np.load(path_filtered)
data_full = np.load(path_full)

ligands_dataset = data_filtered["ligands"]
ligand_distribution_dataset = data_filtered["dataset"]

df_ligands_dataset = pd.DataFrame({
    "Ligand": ligands_dataset,
    "Count": ligand_distribution_dataset,
    "Source": "Filtered"
})

ligands_pdb = data_full["ligands"]
ligand_distribution_pdb = data_full["dataset"]

df_ligands_pdb = pd.DataFrame({
    "Ligand": ligands_pdb,
    "Count": ligand_distribution_pdb,
    "Source": "Retrieved"
})

df_ligands_combined = pd.concat([df_ligands_dataset, df_ligands_pdb])

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(7.5, 3))

sns.set_theme(style="white")
for spine in ['left', 'bottom']:
    axes[0].spines[spine].set_visible(True)
    axes[1].spines[spine].set_visible(True)

source_order = ["Retrieved", "Filtered"]

# Subfigure A: Ligand distribution
ax1 = sns.barplot(
    ax=axes[0],
    x="Ligand",
    y="Count",
    hue="Source",
    data=df_ligands_combined,
    hue_order=source_order,
    saturation=0.9
)
ax1.set_xlabel("")
ax1.set_ylabel("Relative frequency", fontsize=10)
ax1.legend(title=None, fontsize=10)
ax1.tick_params(
    labelsize=10,
    axis='both',
    which='both',
    direction='out',
    length=6,
    width=1,
    bottom=True,
    top=False,
    left=True,
    right=False
)
ax1.set_ylim((0, 0.57))
ax1.text(
    -0.09,
    1.08,
    "A",
    transform=ax1.transAxes,
    fontsize=16,
    fontweight="bold",
    va="top",
    ha="right",
)

# Subfigure B: EC class distribution
ax2 = sns.barplot(
    ax=axes[1],
    x="EC",
    y="Frequency",
    hue="Source",
    data=df_combined,
    hue_order=source_order,
    saturation=0.9
)
ax2.set_xlabel("")
ax2.set_ylabel("Relative frequency", fontsize=10)
ax2.legend(title=None, fontsize=10)
ax2.tick_params(
    labelsize=10,
    axis='both',
    which='both',
    direction='out',
    length=6,
    width=1,
    bottom=True,
    top=False,
    left=True,
    right=False
)
ax2.set_ylim((0, 0.57))
ax2.text(
    -0.12,
    1.08,
    "B",
    transform=ax2.transAxes,
    fontsize=16,
    fontweight="bold",
    va="top",
    ha="right",
)

for ax in [ax1, ax2]:
    for patch in ax.patches:
        patch.set_edgecolor("black")
        patch.set_linewidth(0.5)
    legend = ax.get_legend()
    if legend:
        for handle in legend.legend_handles:
            handle.set_edgecolor("black")
            handle.set_linewidth(0.5)

plt.tight_layout()
plt.savefig(config.directory.figures / "manuscript/Fig2.png", dpi=600)
plt.savefig(config.directory.figures / "manuscript/Fig2.tif", dpi=600)
plt.show()


In [ ]:
# Fig 3 RMSD mode overlap and selection of the network similarity threshold

from __future__ import annotations  # noqa: F404
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib.ticker import LogLocator, LogFormatterMathtext
from config import config

sns.set_style("white")

data = np.load(config.directory.analysis / "rmsd_density.npz")
bin_centers = data["bin_centers"]
hist_density = data["hist_density"]
range_min = data["range_min"]
range_max = data["range_max"]


rmsd_threshold = 0.8

# ---------------------------
# Subfigure A: Mode Overlap Plot
# ---------------------------
data_path = config.directory.analysis / "bimodal_overlap.npy"
gap_distances = np.load(data_path)
ranges = [
    (0.0, 0.1),
    (0.1, 0.2),
    (0.2, 0.3),
    (0.3, 0.4),
    (0.4, 0.5),
    (0.5, 0.6),
    (0.6, 0.7),
    (0.7, 0.8),
    (0.8, 0.9),
    (0.9, 1.0),
    (1.0, 1.1),
    (1.1, 1.2),
]
x_positions = np.arange(1, len(ranges) + 1)
x_labels = [f"{low:.1f}–{high:.1f}" for low, high in ranges]

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(7.5, 3.15))
(ax1, ax2) = axes
ax1.scatter(
    x_positions,
    gap_distances,
    s=25,
    facecolor="0.4",
    edgecolor="black",
    linewidth=0.6,
    marker="o",
    label="GMM Overlap"
)
ax1.set_xticks(x_positions)
ax1.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=6)
ax1.set_xlabel("RMSD range", fontsize=10)
ax1.set_ylabel(r"$\delta$", fontsize=10)
ax1.tick_params(axis="x", labelsize=10)
ax1.tick_params(axis="y", labelsize=10)
ax1.text(-0.05, 1.05, "A", transform=ax1.transAxes, fontsize=16,
         fontweight="bold", va="top", ha="right")
ax1.set_ylim(-0.03, max(gap_distances) * 1.1)
ax1.grid(False)

# --- Subfigure B: RMSD Density ---
sns.lineplot(x=bin_centers, y=hist_density, ax=ax2, color="black", linewidth=0.7)
ax2.fill_between(bin_centers, hist_density, alpha=0.3, color="0.7")
ax2.set_xlabel("RMSD", fontsize=10)
ax2.set_ylabel("Probability density", fontsize=10)
ax2.set_xlim(range_min, 2.5)
ax2.set_ylim(0, 3.3)
ax2.tick_params(axis="x", labelsize=10)
ax2.tick_params(axis="y", labelsize=10)
ax2.axvline(x=rmsd_threshold, color="black", linestyle="--", label="RMSD threshold")
ax2.text(-0.05, 1.05, "B", transform=ax2.transAxes, fontsize=16, fontweight="bold",
         va="top", ha="right")
ax2.grid(False)

# Inset window around the threshold
_zoom_half_width = 0.25
_xmin, _xmax = rmsd_threshold - _zoom_half_width, rmsd_threshold + _zoom_half_width
_xmin = max(_xmin, range_min)
_xmax = min(_xmax, range_max)

# Inset
axins = ax2.inset_axes([0.145, 0.35, 0.28, 0.40])
axins.plot(bin_centers, hist_density, color="black", linewidth=1.0)
axins.fill_between(bin_centers, hist_density, alpha=0.3, color="0.7")
axins.axvline(x=rmsd_threshold, color="black", linestyle="--", linewidth=0.6)
axins.set_xlim(_xmin, _xmax)

_mask = (bin_centers >= _xmin) & (bin_centers <= _xmax)
_local_slice = hist_density[_mask]
_local_max = float(_local_slice.max()) if _mask.any() else float(hist_density.max())
_positive = _local_slice[_local_slice > 0]
_local_min_pos = float(_positive.min()) if _positive.size > 0 else 1e-12

# Log10 y-axis with decade-bounded limits and explicit locators
axins.set_yscale("log", base=10)
ymin_target = _local_min_pos * 0.6
ymax_target = _local_max * 4.0
ymin_dec = 10 ** np.floor(np.log10(ymin_target))
ymax_dec = 10 ** np.ceil(np.log10(ymax_target))
if ymin_dec >= ymax_dec:
    ymax_dec = 10 ** (np.ceil(np.log10(ymin_dec)) + 1)
axins.yaxis.set_major_locator(LogLocator(base=10.0, subs=(1.0,), numticks=10))
axins.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1, numticks=100))
axins.yaxis.set_major_formatter(LogFormatterMathtext())
axins.set_ylim(ymin_dec, ymax_dec)

axins.set_xlabel("")
axins.set_ylabel("")
#axins.set_ylabel(r"$\log_{10}$ probability density", fontsize=6)
axins.tick_params(axis="both", labelsize=6)
axins.grid(False)

# Get coordinates of inset in display space
inv = ax2.transData.inverted()
inset_bbox = axins.get_window_extent(fig.canvas.get_renderer()).transformed(inv)

# Bottom corners of inset in data coordinates
x_inset_min, y_inset_min = inset_bbox.x0, inset_bbox.y0
x_inset_max, y_inset_max = inset_bbox.x1, inset_bbox.y0

# Draw two connecting lines (from inset lower corners to corresponding x-limits)
ax2.plot([_xmin, x_inset_min], [0, y_inset_min],
         color="0.15", linewidth=0.7, linestyle=(0, (2, 2)))
ax2.plot([_xmax, x_inset_max], [0, y_inset_max],
         color="0.15", linewidth=0.7, linestyle=(0, (2, 2)))

for _ax in (ax1, ax2, axins):
    for _sp in _ax.spines.values():
        _sp.set_linewidth(1.2)

plt.tight_layout()
plt.savefig(config.directory.figures / "manuscript/Fig3.png", dpi=600)
plt.savefig(config.directory.figures / "manuscript/Fig3.tif", dpi=600)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import pandas as pd
import numpy as np
from matplotlib.colors import ListedColormap, BoundaryNorm
from config import config

name = "MBSNetwork"
distribution_path = config.directory.networks / f"{name}/ligand_to_ligand_distribution.npz"
zscore_path = config.directory.networks / f"{name}/ligand_zscores.npz"

# Load data
distribution_data = np.load(distribution_path)
metal_distribution = distribution_data["distribution"]
ligand_pdb_ids = [ligand.capitalize() for ligand in distribution_data["ligands"]]

df = pd.DataFrame(metal_distribution.T, columns=ligand_pdb_ids)
df.insert(0, "Ligand", ligand_pdb_ids)
df.iloc[:, 1:] = df.iloc[:, 1:].div(df.iloc[:, 1:].sum(axis=1), axis=0)

zscore_data = np.load(zscore_path)
z_scores = zscore_data["z_scores"]
ligands = [x.capitalize() for x in zscore_data["ligands"]]
p_adjusted = zscore_data["p_adjusted"]
ns_mask = (p_adjusted > 0.05)

default_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
ligand_colors = [default_colors[i % len(default_colors)] for i in range(len(ligands))]

colors = ["#D9A500", "#000000", "#b2182b"]
bounds = [np.min(z_scores), -1.96, 1.96, np.max(z_scores)]
cmap = ListedColormap(colors)
norm = BoundaryNorm(bounds, len(colors))

scale = 7.5 / 20.0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.5, 8 * scale))

df.plot(
    x="Ligand",
    kind="bar",
    stacked=True,
    rot=0,
    edgecolor="k",
    linewidth=0.4,
    width=0.7,
    ax=ax1,
    legend=False,
    fontsize=14 * scale
)
ax1.set_xlabel("")
ax1.set_ylabel("")
ax1.tick_params(axis="x", labelsize=8)
ax1.tick_params(axis="y", labelsize=8)
ax1.text(-0.06, 1.05, 'A', transform=ax1.transAxes, fontsize=16, fontweight='bold', va='top', ha='right')

zero_mask_raw = (df.iloc[:, 1:].to_numpy() == 0.0)
zero_mask = np.rot90(zero_mask_raw, k=1)

z_for_plot = z_scores.copy()
z_for_plot[p_adjusted > 0.05] = 0.0

cmap.set_bad(color='white')
z_masked = np.ma.array(z_for_plot, mask=zero_mask)

im = ax2.imshow(z_masked, cmap=cmap, norm=norm, aspect="auto")
ax2.text(-0.06, 1.05, 'B', transform=ax2.transAxes, fontsize=16, fontweight='bold', va='top', ha='right')

legend_handles = [
    Patch(color="#b2182b", label="Over-enriched"),
    Patch(color="#D9A500", label="Under-enriched"),
    Patch(color="#000000", label="Not significant"),
    Patch(facecolor="white", edgecolor="black", linewidth=0.8 * scale, label="Not observed"),
]


ax2.legend(
    handles=legend_handles,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.1),
    ncol=4,
    frameon=False,
    fontsize=15 * scale,
    handlelength=1.2,
    handleheight=1.2,
    columnspacing=1.5
)

for i, (ligand_x, ligand_y, color_x, color_y) in enumerate(zip(ligands, reversed(ligands), ligand_colors, reversed(ligand_colors))):
    ax2.add_patch(plt.Rectangle((i - 0.5, z_scores.shape[0] - 0.5 + 0.01), 1, 0.35,
                                facecolor=color_x, edgecolor='black', linewidth=0.3 * scale,
                                clip_on=False, transform=ax2.transData))
    ax2.text(i, z_scores.shape[0] - 0.5 + 0.185, ligand_x,
             ha='center', va='center', fontsize=14 * scale,
             color='black', rotation=0, clip_on=False, transform=ax2.transData)

    ax2.add_patch(plt.Rectangle((-0.5 - 0.36, i - 0.5), 0.35, 1,
                                facecolor=color_y, edgecolor='black', linewidth=0.3 * scale,
                                clip_on=False, transform=ax2.transData))
    ax2.text(-0.5 - 0.18, i, ligand_y,
             ha='center', va='center', fontsize=14 * scale,
             color='black', rotation=0, clip_on=False, transform=ax2.transData)

ax2.set_xticklabels([])
ax2.set_yticklabels([])
ax2.tick_params(axis='x', which='major', length=0)
ax2.tick_params(axis='y', which='major', length=0)

ax2.set_xticks(np.arange(z_scores.shape[1]+1)-0.5, minor=True)
ax2.set_yticks(np.arange(z_scores.shape[0]+1)-0.5, minor=True)
ax2.grid(which="minor", color="black", linewidth=1.0 * scale, alpha=0.7)
ax2.tick_params(which='minor', bottom=False, left=False)

ax2.invert_yaxis()
ax2.xaxis.set_ticks_position('top')
ax2.xaxis.set_label_position('top')

plt.tight_layout()
fig.savefig(config.directory.figures / "manuscript/Fig5.png", dpi=600)
fig.savefig(config.directory.figures / "manuscript/Fig5.tif", dpi=600)
plt.show()


In [ ]:
# Fig 6 Enrichment of EC subclass co-occurrence among network links

import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, TwoSlopeNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

from config import config

name = "MBSNetwork"
digit = 3

z_path = config.directory.networks / f"{name}/ec_zscores_{digit}.npz"
dist_path = config.directory.networks / f"{name}/ec_to_ec_distribution_{digit}.npz"

z_data = np.load(z_path, allow_pickle=True)
z_scores = z_data["z_scores"]
p_adjusted = z_data["p_adjusted"]
ec_numbers = z_data["ec_numbers"]

ns_mask = (p_adjusted > 0.05)
z_for_plot = z_scores.copy()
z_for_plot[ns_mask] = 0.0

colors = ["#f6fa00", "#000000", "#b2182b"]  # low, mid, high
bounds = [np.nanmin(z_scores), -1.96, 1.96, np.nanmax(z_scores)]
cmap = ListedColormap(colors)
norm = TwoSlopeNorm(vmin=-30, vcenter=0, vmax=30)

fig, ax = plt.subplots(figsize=(7.5, 6))

im = ax.imshow(z_for_plot, cmap=cmap, norm=norm, aspect="auto")

ax.invert_yaxis()
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")
ax.set_xticks([])
ax.set_yticks([])

n = z_scores.shape[0]
ax.set_xticks(np.arange(n + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(n + 1) - 0.5, minor=True)
ax.grid(which="minor", color="black", linewidth=1.0, alpha=0.7)
ax.tick_params(which="minor", bottom=False, left=False)

ax_pos = ax.get_position()
x0, y0, width, height = ax_pos.x0, ax_pos.y0, ax_pos.width, ax_pos.height

for ec_class in range(1, 8):
    pattern = re.compile(rf"^{ec_class}\.")
    indices = [i for i, ec in enumerate(ec_numbers) if pattern.match(str(ec))]
    if not indices:
        continue

    start = min(indices)
    end = max(indices)
    center = (start + end) / 2
    label = f"EC {ec_class}"

    x_center = x0 + width * (center / n)
    fig.text(
        x_center,
        y0 + height + 0.04,
        label,
        ha="center",
        va="bottom",
        fontsize=10,
        rotation=90,
    )

    x_start = x0 + width * (start / n)
    x_end = x0 + width * ((end + 1) / n)
    y_bracket_top = y0 + height + 0.03

    fig.add_artist(Line2D([x_start, x_start], [y_bracket_top, y_bracket_top - 0.005],
                          transform=fig.transFigure, color="black", lw=1))
    fig.add_artist(Line2D([x_end, x_end], [y_bracket_top, y_bracket_top - 0.005],
                          transform=fig.transFigure, color="black", lw=1))
    fig.add_artist(Line2D([x_start, x_end], [y_bracket_top, y_bracket_top],
                          transform=fig.transFigure, color="black", lw=1))

    flipped_center = n - 1 - center
    y_center = y0 + height * (flipped_center / n)
    fig.text(
        x0 - 0.06,
        y_center,
        label,
        ha="right",
        va="center",
        fontsize=10,
    )

    flipped_start = n - 1 - end
    flipped_end = n - 1 - start

    y_start = y0 + height * (flipped_start / n)
    y_end = y0 + height * ((flipped_end + 1) / n)
    x_bracket_left = x0 - 0.03

    fig.add_artist(Line2D([x_bracket_left, x_bracket_left - 0.005], [y_start, y_start],
                          transform=fig.transFigure, color="black", lw=1))
    fig.add_artist(Line2D([x_bracket_left, x_bracket_left - 0.005], [y_end, y_end],
                          transform=fig.transFigure, color="black", lw=1))
    fig.add_artist(Line2D([x_bracket_left - 0.005, x_bracket_left - 0.005], [y_start, y_end],
                          transform=fig.transFigure, color="black", lw=1))

legend_handles = [
    Patch(color="#b2182b", label="Over-enriched"),
    Patch(color="#D9A500", label="Under-enriched"),
    Patch(color="#000000", label="Not significant"),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.04),
    ncol=4,
    frameon=False,
    fontsize=10,
    handlelength=1.2,
    handleheight=1.2,
)

plt.axis("off")
fig.savefig(config.directory.figures / "manuscript/Fig6.png", dpi=600)
fig.savefig(config.directory.figures / "manuscript/Fig6.tif", dpi=600)
plt.show()


In [ ]:
# Fig 10 Sequence-geometry decoupling highlights recurrent metal-binding site geometries across sequence-divergent proteins

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr
from mpl_toolkits.axes_grid1 import make_axes_locatable

from analysis.sequence import get_edge_dict
from config import config

name = "MBSNetwork"
path = config.directory.networks / f"{name}/edge_attributes_with_seq_identity.tsv"
edge_dict = get_edge_dict(path)

x = 1 - np.array([edge["rmsd"] / 0.8 for edge in edge_dict.values()], dtype=float)
y = np.array([edge["identity"] for edge in edge_dict.values()], dtype=float)

rho, pval = spearmanr(x, y)

print("Spearman correlation between normalized geometric similarity and global sequence identity")
print(f"ρ = {rho:.4f}")
print(f"p-value = {pval:.3e}")
print(f"n = {len(x)}")

results = np.load(
    config.directory.networks / f"{name}/geometry_conserved_sequence_divergent_pairs.npy"
)

xB = results[:, 0].astype(float)
E = results[:, 1].astype(float)

logEB = np.log10(E)

logE_threshold = np.log10(1e-3)

identity_cutoff = 25.0
norm_geom_cutoff = 0.375
candidate_mask = (y < identity_cutoff) & (x > norm_geom_cutoff)

fig, axes = plt.subplots(1, 2, figsize=(7.5, 2.7))

ax = axes[0]

hb = ax.hexbin(x, y, gridsize=100, cmap="viridis", bins="log")
cb = fig.colorbar(hb, ax=ax)
cb.set_label("Log$_{10}$(Count)", fontsize=7)
cb.ax.tick_params(labelsize=6)

ax.set_xlabel(r"$1 - \frac{\mathrm{RMSD}}{\max(\mathrm{RMSD})}$", fontsize=8)
ax.set_ylabel("Global sequence identity (%)", fontsize=8)
ax.tick_params(labelsize=6)

highlight = plt.Rectangle(
    (norm_geom_cutoff, 0),
    1 - norm_geom_cutoff,
    identity_cutoff,
    linewidth=0.75,
    edgecolor="red",
    facecolor="none",
    linestyle="--",
)
ax.add_patch(highlight)

divider = make_axes_locatable(ax)

ax_top = divider.append_axes("top", size="22%", pad=0.15, sharex=ax)
ax_top.hist(x, bins=50, color="orange", edgecolor="black")
ax_top.set_yscale("log")
ax_top.set_ylabel("Count", fontsize=6)
ax_top.tick_params(axis="x", labelbottom=False)
ax_top.tick_params(axis="y", labelsize=5)
ax_top.spines["right"].set_visible(False)
ax_top.spines["top"].set_visible(False)

ax_right = divider.append_axes("right", size="22%", pad=0.15, sharey=ax)
ax_right.hist(y, bins=50, orientation="horizontal", color="orange", edgecolor="black")
ax_right.set_xscale("log")
ax_right.set_xlabel("Count", fontsize=6)
ax_right.tick_params(axis="y", labelleft=False)
ax_right.tick_params(axis="x", labelsize=5)
ax_right.spines["top"].set_visible(False)
ax_right.spines["right"].set_visible(False)

ax2 = axes[1]

mask_homology = logEB < logE_threshold
ax2.scatter(
    xB[mask_homology], logEB[mask_homology],
    s=5, alpha=0.6, color="grey", edgecolors="none",
    label=r"$E < 10^{-3}$",
)
ax2.scatter(
    xB[~mask_homology], logEB[~mask_homology],
    s=11, alpha=0.9, color="red", edgecolors="none",
    label=r"$E >= 10^{-3}$",
)
ax2.set_ylim([-10, 1.05])

ax2.axhline(logE_threshold, linestyle="--", linewidth=0.75, color="red")

ax2.set_xlabel(r"$1 - \frac{\mathrm{RMSD}}{\max(\mathrm{RMSD})}$", fontsize=8)
ax2.set_ylabel(r"$\log_{10}(\mathrm{E\text{-}value})$", fontsize=8)
ax2.tick_params(labelsize=6)

fig.text(
    0.005, 1, "A",
    fontsize=16,
    fontweight="bold",
    va="top",
    ha="left",
)

fig.text(
    0.52, 1, "B",
    fontsize=16,
    fontweight="bold",
    va="top",
    ha="left",
)

plt.tight_layout()
plt.savefig(config.directory.figures / "manuscript/Fig10.png", dpi=600)
plt.savefig(config.directory.figures / "manuscript/Fig10.tif", dpi=600)
plt.show()


In [ ]:
# Fig 11 Network enrichment and structural proximity filters for drug off-target prediction

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from config import config

analysis_dir = config.directory.analysis
volcano_csv = analysis_dir / "drugs/volcano_data.csv"
off_targets_csv = analysis_dir / "drugs/predicted_off_targets.csv"

df = pd.read_csv(volcano_csv)
off_target_drugs = pd.read_csv(off_targets_csv)["drug"].astype(str).tolist()

df["drug"] = df["drug"].astype(str)
df["is_significant"] = df["p_adjusted"] < 0.05
df["is_off_target"] = df["drug"].isin(off_target_drugs)

base_mask = ~df["is_significant"] & ~df["is_off_target"]
sig_mask  = df["is_significant"] & ~df["is_off_target"]
ot_mask   = df["is_off_target"]

fig, ax = plt.subplots(figsize=(6.5, 4.5))

color_base = "#B0B0B0"
color_sig  = plt.rcParams["axes.prop_cycle"].by_key()["color"][0]
color_ot   = plt.rcParams["axes.prop_cycle"].by_key()["color"][1]
marker_ot  = "s"
y_thr = -np.log10(0.05)

h_base = ax.scatter(
    df.loc[base_mask, "log2_enrichment"],
    df.loc[base_mask, "log10_p"],
    s=12, alpha=0.60, linewidths=0, c=color_base, label="FDR > 0.05", zorder=1
)
h_sig = ax.scatter(
    df.loc[sig_mask, "log2_enrichment"],
    df.loc[sig_mask, "log10_p"],
    s=14, alpha=0.85, linewidths=0, c=color_sig, label="FDR < 0.05", zorder=2
)
h_ot = ax.scatter(
    df.loc[ot_mask, "log2_enrichment"],
    df.loc[ot_mask, "log10_p"],
    s=25, alpha=0.95, c=color_ot, marker=marker_ot, edgecolors="black", linewidths=0.6,
    label="FDR < 0.05 & proximal binding", zorder=3
)

ax.axhline(y_thr, color="0.5", linestyle="--", linewidth=1, zorder=0)
ax.axvline(0, color="0.2", linestyle="--", linewidth=1, zorder=0)

ax.set_xlabel(r"$\log_2$(Enrichment ratio)")
ax.set_ylabel(r"$-\log_{10}$(Empirical $p$)")
ax.margins(x=0.05, y=0.05)


# Annotate specific highlighted drugs with their DrugBank IDs
highlight_drugs = ["DB07954", "DB00786", "DB02255", "DB03880", "DB02909"]
drug_names = ["IBMX", "Marimastat", "Ilomastat", "Batimastat", "CF2A"]
id_to_name = dict(zip(highlight_drugs, drug_names))
highlight_df = df[df["drug"].isin(highlight_drugs)]
annot_pos = {
    "DB07954": {"xytext": (-25, -70), "ha": "center"},
    "DB00786": {"xytext": ( -10, -90), "ha": "center"},
    "DB02255": {"xytext": ( -20, -60), "ha": "center"},
    "DB03880": {"xytext": (-120, -10), "ha": "center"},
    "DB02909": {"xytext": (-10, -60), "ha": "center"},
}
for _, row in highlight_df.iterrows():
    cfg = annot_pos.get(row["drug"], {})
    xytext = cfg.get("xytext", (0, -70))   # fallback if a drug isn't in annot_pos
    ha = cfg.get("ha", "center")
    ax.annotate(
        id_to_name.get(row["drug"], row["drug"]),
        (row["log2_enrichment"], row["log10_p"]),
        xytext=xytext, textcoords="offset points",
        ha=ha, va="top",
        fontsize=8, color="black",
        arrowprops=dict(arrowstyle="->", lw=0.9, color="black", alpha=0.9),
        zorder=6
    )

fig.subplots_adjust(right=0.78)
leg = ax.legend(
    handles=[h_base, h_sig, h_ot],
    frameon=True, facecolor="white", edgecolor="0.8",
    fontsize=9, loc="lower right", borderaxespad=0.6
)
leg.get_frame().set_linewidth(0.8)

fig.savefig(config.directory.figures / "manuscript/Fig11.png", dpi=600, bbox_inches="tight")
fig.savefig(config.directory.figures / "manuscript/Fig11.tif", dpi=600, bbox_inches="tight")
plt.show()


In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import seaborn as sns
from config import config

alignments = config.directory.alignments / "preliminary/random_alignments.npy"
data = np.load(alignments)
rmsd_scores = data[:, 2]

bin_width = 0.1
bins = np.arange(0, rmsd_scores.max(), bin_width)

fig, ax = plt.subplots(figsize=(6, 4))

sns.histplot(
    ax=ax,
    data=rmsd_scores,
    bins=bins,
    color="0.6",
    stat="probability"
)

ax.set_xlabel("RMSD", fontsize=12)
ax.set_ylabel("Relative frequency", fontsize=12)
ax.set_xlim(0, 2.5)

for patch in ax.patches:
    patch.set_edgecolor("black")
    patch.set_linewidth(0.8)

ax.tick_params(direction="out", length=4, width=0.8, labelsize=12)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1)

fig.tight_layout()
fig.savefig(config.directory.figures / "manuscript/Fig_S1.png", dpi=600, bbox_inches="tight")
plt.show()


In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import seaborn as sns
from config import config

arr_path = config.directory.alignments / "preliminary/global_registration.npy"
arr = np.load(arr_path)

rmsd_scores = arr[:, 2]
rmsd_scores = rmsd_scores[np.isfinite(rmsd_scores)]

bin_width = 0.1
bins = np.arange(0, rmsd_scores.max(), bin_width)

fig, ax = plt.subplots(figsize=(6, 4))

sns.histplot(
    ax=ax,
    data=rmsd_scores,
    bins=bins,
    color="0.6",
    stat="probability",
)

ax.set_xlabel("RMSD", fontsize=12)
ax.set_ylabel("Relative frequency", fontsize=12)
ax.set_xlim(0, 2.5)

for patch in ax.patches:
    patch.set_edgecolor("black")
    patch.set_linewidth(0.8)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1)

fig.tight_layout()
fig.savefig(config.directory.figures / "manuscript/Fig_S2.png", dpi=600, bbox_inches="tight")
plt.show()


In [ ]:
from collections import Counter
import seaborn as sns
from matplotlib import pyplot as plt
import pandas as pd
from config import config

name = "MBSNetwork"
path = config.directory.networks / f"{name}/geometry_conserved_sequence_divergent_pairs.tsv"

ligand_counts = Counter()
with path.open() as f:
    next(f)
    for line in f:
        parts = line.rstrip("\n").split("\t")
        lig1 = parts[1].strip() if len(parts) > 1 else ""
        lig2 = parts[8].strip() if len(parts) > 8 else ""
        if lig1:
            ligand_counts[lig1] += 1
        if lig2:
            ligand_counts[lig2] += 1

norm_counts = Counter({k.strip().upper(): v for k, v in ligand_counts.items() if k})

order_labels = ["W", "Fe", "Co", "Cu", "Ni", "Zn", "Mn", "Mo", "V"]
order_keys   = ["W", "FE", "CO", "CU", "NI", "ZN", "MN", "MO", "V"]

counts = [norm_counts.get(k, 0) for k in order_keys]
total = sum(norm_counts.values())
rel = [(c / total) if total > 0 else 0.0 for c in counts]

df_ligands = pd.DataFrame(
    {"Ligand": order_labels, "Count": counts, "Relative": rel}
)

fig, ax = plt.subplots(figsize=(6, 4))

sns.set_theme(style="white")
for spine in ["left", "bottom"]:
    ax.spines[spine].set_visible(True)

ax1 = sns.barplot(
    ax=ax,
    x="Ligand",
    y="Relative",
    data=df_ligands,
    order=order_labels,
    saturation=0.9
)

ax1.set_xlabel("")
ax1.set_ylabel("Relative frequency", fontsize=10)
ax1.tick_params(
    labelsize=10,
    axis="both",
    which="both",
    direction="out",
    length=6,
    width=1,
    bottom=True,
    top=False,
    left=True,
    right=False,
)
ax1.set_ylim(top=0.5)

for patch in ax1.patches:
    patch.set_edgecolor("black")
    patch.set_linewidth(0.8)

for spine in ax1.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1)

plt.tight_layout()
plt.savefig(config.directory.figures / "manuscript/Fig_S4.png", dpi=600)
plt.show()
